This notebook demonstrates how to run the forest deforestation User Defined Process

In [ ]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

In [ ]:
spatial_extent = {
    "west": 30.55,
    "south": 1.07,
    "east": 31.23,
    "north": 1.55,
}

resample_spatial_resolution = 30  # m

# Load decimal year of deforestation

In [ ]:
# load results from previous batch job
JOB_ID = "j-26080615434744c6985547c10c25c56f"

deforestation_year = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
deforestation_year = deforestation_year.drop_dimension("t")

# load forest baseline

In [ ]:
# load results from previous batch job
JOB_ID = "j-2609111429014f79ba27580bfcfc33eb"

forest_baseline_mask = connection.load_stac_from_job(
    JOB_ID,
    spatial_extent=spatial_extent,
)

In [ ]:
# load_stac adds a time dimension 😠
forest_baseline_mask = forest_baseline_mask.drop_dimension("t")

In [ ]:
# load_stac incorrectly sets nodata=0 😠
# https://forum.dataspace.copernicus.eu/t/round-trip-nodata/5512
forest_baseline_mask = forest_baseline_mask == 1

# Run KPIs UDP

In [ ]:
kpis_vector_cube = connection.datacube_from_process(
    "KPIs",
    namespace=urls.KPIS_UDP,
    forest_baseline_datacube=forest_baseline_mask,
    decimal_year_of_deforestation_datacube=deforestation_year,
    spatial_extent=spatial_extent,
)

In [ ]:
job = kpis_vector_cube.create_job(out_format="Parquet")

In [ ]:
job.start_and_wait()

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output-udp/
!rm -r output-udp/

In [ ]:
results.download_files("output-udp/")

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)